In [75]:
import pandas as pd

In [76]:
# ================================================
# Step 1 — Understand Data
# ================================================

In [77]:
# Load sales data
sales = pd.read_csv('../data/Online Retail Sales Data.csv')

# Load campaign data
campaign = pd.read_csv('../data/Campaign Response.csv')

print(sales.shape)
print(campaign.shape)

(337321, 8)
(3834, 6)


In [78]:
# Look at first 5 rows of sales data
print("=== SALES DATA ===")
print(sales.head())
print("\nColumns:", sales.columns.tolist())



=== SALES DATA ===
   CustomerID InvoiceNo StockCode                Description  Quantity  \
0       13313    539993     22386    JUMBO BAG PINK POLKADOT        10   
1       13313    539993     21499         BLUE POLKADOT WRAP        25   
2       13313    539993     21498        RED RETROSPOT WRAP         25   
3       13313    539993     22379   RECYCLING BAG RETROSPOT          5   
4       13313    539993     20718  RED RETROSPOT SHOPPER BAG        10   

      InvoiceDate  UnitPrice         Country  
0  04/01/23 10:00       1.95  United Kingdom  
1  04/01/23 10:00       0.42  United Kingdom  
2  04/01/23 10:00       0.42  United Kingdom  
3  04/01/23 10:00       2.10  United Kingdom  
4  04/01/23 10:00       1.25  United Kingdom  

Columns: ['CustomerID', 'InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'Country']


In [79]:
# Look at first 5 rows of campaign data
print("\n=== CAMPAIGN DATA ===")
print(campaign.head())
print("\nColumns:", campaign.columns.tolist())


=== CAMPAIGN DATA ===
   CustomerID  response  n_comp  loyalty nps  n_communications
0       12346         1       2        1   7                 8
1       12747         0       1        1   3                 3
2       12748         1       0        1   9                 6
3       12749         0       4        1   2                 5
4       12820         0       4        1   2                 2

Columns: ['CustomerID', 'response', 'n_comp', 'loyalty', 'nps', 'n_communications']


In [80]:
# Check data types and missing values
print("=== SALES INFO ===")
print(sales.info())



=== SALES INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 337321 entries, 0 to 337320
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   CustomerID   337321 non-null  int64  
 1   InvoiceNo    337321 non-null  object 
 2   StockCode    337321 non-null  object 
 3   Description  337321 non-null  object 
 4   Quantity     337321 non-null  int64  
 5   InvoiceDate  337321 non-null  object 
 6   UnitPrice    337321 non-null  float64
 7   Country      337321 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 20.6+ MB
None


In [81]:
print("\n=== CAMPAIGN INFO ===")
print(campaign.info())


=== CAMPAIGN INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3834 entries, 0 to 3833
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   CustomerID        3834 non-null   int64 
 1   response          3834 non-null   int64 
 2   n_comp            3834 non-null   int64 
 3   loyalty           3834 non-null   int64 
 4   nps               3834 non-null   object
 5   n_communications  3834 non-null   int64 
dtypes: int64(5), object(1)
memory usage: 179.8+ KB
None


In [82]:
# Check missing values
print("=== MISSING VALUES ===")
print("Sales missing:\n", sales.isnull().sum())
print("\nCampaign missing:\n", campaign.isnull().sum())

=== MISSING VALUES ===
Sales missing:
 CustomerID     0
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
Country        0
dtype: int64

Campaign missing:
 CustomerID          0
response            0
n_comp              0
loyalty             0
nps                 0
n_communications    0
dtype: int64


In [83]:
# Check negative quantities (returns/cancellations)
print("Negative quantities:", (sales['Quantity'] < 0).sum())
print("Zero quantities:", (sales['Quantity'] == 0).sum())

Negative quantities: 6939
Zero quantities: 0


In [84]:
# Check negative prices
print("Negative prices:", (sales['UnitPrice'] <= 0).sum())


Negative prices: 23


In [85]:
# Check duplicates
print("Sales duplicates:", sales.duplicated().sum())
print("Campaign duplicates:", campaign.duplicated().sum())

Sales duplicates: 4657
Campaign duplicates: 0


In [86]:
# 1. CLEANING DATA
# 1.1 Remove negative quantities

sales = sales[sales['Quantity'] > 0]


In [87]:
# 1.2. Remove negative or zero prices
sales = sales[sales['UnitPrice'] > 0]

In [88]:
# 1.3 Remove duplicate rows
sales = sales.drop_duplicates()


In [89]:
# 2. Change Data Types
# 2.1 Convert InvoiceDate from text to date
sales['InvoiceDate'] = pd.to_datetime(sales['InvoiceDate'],
                                       format='%d/%m/%y %H:%M')
print("InvoiceDate:", sales['InvoiceDate'].dtype)


InvoiceDate: datetime64[ns]


In [90]:
# 2.2 Fix nps - convert from text to number
campaign['nps'] = pd.to_numeric(campaign['nps'], errors='coerce')
print("nps:", campaign['nps'].dtype)

nps: float64


In [91]:
# ================================================
# Step 2 — Create Features
# ================================================



In [92]:
# First calculate TotalSales per row
sales['TotalSales'] = sales['Quantity'] * sales['UnitPrice']

# Feature 1 - Total Spending per customer
Total_Spending = sales.groupby('CustomerID')['TotalSales'].sum().reset_index()
Total_Spending.columns = ['CustomerID', 'Total_Spending']

# Feature 2 - Purchase Frequency (number of invoices)
Purchase_Frequency = sales.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
Purchase_Frequency.columns = ['CustomerID', 'Purchase_Frequency']

# Feature 3 - Recency (days since last purchase)
max_date = sales['InvoiceDate'].max()
Recency = sales.groupby('CustomerID')['InvoiceDate'].max().reset_index()
Recency['Recency'] = (max_date - Recency['InvoiceDate']).dt.days
Recency = Recency[['CustomerID', 'Recency']]

# Feature 4 - Unique Products (based on StockCode)
Unique_Products = sales.groupby('CustomerID')['StockCode'].nunique().reset_index()
Unique_Products.columns = ['CustomerID', 'Unique_Products']

# Feature 5 - Avg Basket Size (total spending / purchase frequency)
Avg_Basket_Size = Total_Spending.merge(Purchase_Frequency, on='CustomerID')
Avg_Basket_Size['Avg_Basket_Size'] = (
    Avg_Basket_Size['Total_Spending'] / Avg_Basket_Size['Purchase_Frequency']
)
Avg_Basket_Size = Avg_Basket_Size[['CustomerID', 'Avg_Basket_Size']]

print(Total_Spending.head(10))
print("\nFeature 2 - Purchase Frequency:")
print(Purchase_Frequency.head(10))
print("\nFeature 3 - Recency:")
print(Recency.head(10))
print("\nFeature 4 - Unique Products:")
print(Unique_Products.head(10))
print("\nFeature 5 - Avg Basket Size:")
print(Avg_Basket_Size.head(10))




   CustomerID  Total_Spending
0       12346        77183.60
1       12747         3489.74
2       12748        28868.19
3       12749         4090.88
4       12820          942.34
5       12821           92.72
6       12822          948.88
7       12823         1759.50
8       12824          397.12
9       12826         1319.72

Feature 2 - Purchase Frequency:
   CustomerID  Purchase_Frequency
0       12346                   1
1       12747                   9
2       12748                 174
3       12749                   5
4       12820                   4
5       12821                   1
6       12822                   2
7       12823                   5
8       12824                   1
9       12826                   6

Feature 3 - Recency:
   CustomerID  Recency
0       12346      325
1       12747        1
2       12748        0
3       12749        3
4       12820        2
5       12821      213
6       12822       70
7       12823       74
8       12824       59
9       128

In [93]:
# ================================================
# Merge All Features
# ================================================

features = Total_Spending\
    .merge(Purchase_Frequency, on='CustomerID')\
    .merge(Recency,            on='CustomerID')\
    .merge(Unique_Products,    on='CustomerID')\
    .merge(Avg_Basket_Size,    on='CustomerID')

print("Shape:", features.shape)
print("\nColumns:", features.columns.tolist())
print("\nFirst 5 rows:")
print(features.head())
print("\nStats:")
print(features.describe().round(2))

Shape: (3812, 6)

Columns: ['CustomerID', 'Total_Spending', 'Purchase_Frequency', 'Recency', 'Unique_Products', 'Avg_Basket_Size']

First 5 rows:
   CustomerID  Total_Spending  Purchase_Frequency  Recency  Unique_Products  \
0       12346        77183.60                   1      325                1   
1       12747         3489.74                   9        1               39   
2       12748        28868.19                 174        0             1602   
3       12749         4090.88                   5        3              160   
4       12820          942.34                   4        2               55   

   Avg_Basket_Size  
0     77183.600000  
1       387.748889  
2       165.909138  
3       818.176000  
4       235.585000  

Stats:
       CustomerID  Total_Spending  Purchase_Frequency  Recency  \
count     3812.00         3812.00             3812.00  3812.00   
mean     15551.81         1780.71                4.03    83.54   
std       1575.75         7126.17              

In [95]:
#================================================
# Task 5 - Create Master Dataset
# ================================================

master = campaign.merge(features, on='CustomerID', how='left')

print("Master dataset shape:", master.shape)
print("\nColumns:", master.columns.tolist())
print("\nFirst 5 rows:")
print(master.head())
print("\nMissing values:")
print(master.isnull().sum())

Master dataset shape: (3834, 11)

Columns: ['CustomerID', 'response', 'n_comp', 'loyalty', 'nps', 'n_communications', 'Total_Spending', 'Purchase_Frequency', 'Recency', 'Unique_Products', 'Avg_Basket_Size']

First 5 rows:
   CustomerID  response  n_comp  loyalty  nps  n_communications  \
0       12346         1       2        1  7.0                 8   
1       12747         0       1        1  3.0                 3   
2       12748         1       0        1  9.0                 6   
3       12749         0       4        1  2.0                 5   
4       12820         0       4        1  2.0                 2   

   Total_Spending  Purchase_Frequency  Recency  Unique_Products  \
0        77183.60                 1.0    325.0              1.0   
1         3489.74                 9.0      1.0             39.0   
2        28868.19               174.0      0.0           1602.0   
3         4090.88                 5.0      3.0            160.0   
4          942.34                 4.0   